CHECK EMBEDDINGS

In [ ]:
import numpy as np

embeddings = np.load("/home/tbak/KBLaM/nano_data/synthetic_data_BigOAI_embd_key.npy")

In [ ]:
print("Shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

In [ ]:
print(embeddings[0])         # First embedding vector
print(embeddings[0][:10])    # First 10 dimensions of the first vector
print(embeddings[100])       # 101st vector

CHECK MODEL

In [ ]:
from transformers import AutoTokenizer
from src.kblam.models.phi3_model import KBLaMPhi3ForCausalLM  # or wherever it's defined
from src.kblam.kb_encoder import KBEncoder
from src.kblam.models.kblam_config import KBLaMConfig
import torch
from pathlib import Path

def load_kblam_model(
    model_path: str,
    encoder_path: str,
    llm_type: str = "phi3",
    encoder_spec: str = "BigOAI",
    kb_layer_frequency: int = 2,
    kb_scale_factor: float = 1.0,
    query_head_path: str = None,
):
    model_path = Path(model_path)
    encoder_path = Path(encoder_path)

    if not model_path.exists():
        raise FileNotFoundError(f"Model directory not found: {model_path.resolve()}")

    tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-4k-instruct", trust_remote_code=True, padding_side="left")
    tokenizer.pad_token = "^"

    if llm_type.lower() == "llama3":
        from src.kblam.models.llama3_model import KblamLlamaForCausalLM  # adjust path
        model = KblamLlamaForCausalLM.from_pretrained(
            model_path,
            device_map="cuda",
            torch_dtype="auto",
            trust_remote_code=True,
            local_files_only=True  # <--- ensures it never goes online
        )
        if query_head_path:
            model.load_query_head(query_head_path)
    else:
        model = KBLaMPhi3ForCausalLM.from_pretrained(
            model_path,
            device_map="cuda",
            torch_dtype="auto",
            trust_remote_code=True,
            local_files_only=True  # <--- ensures it never goes online
        )

    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.eval()

    # Build matching KB config
    kb_config = KBLaMConfig(
        sep_query_head=True,
        kb_layer_frequency=kb_layer_frequency,
        kb_scale_factor=kb_scale_factor,
    )

    # Recreate encoder
    encoder = KBEncoder(
        encoder_name=encoder_spec,
        projector_type="linear",  # Match training setting
        endpoint_url="",
        out_dim=model.config.hidden_size * (model.config.num_hidden_layers // kb_layer_frequency + 1),
        frozen_base_model=True,
        projector_kwargs={"mlp_depth": 1, "mlp_hidden_dim": 512},
        device=torch.device("cuda"),
        get_oai_embd_online=True,
    )
    encoder.load_state_dict(torch.load(encoder_path))

    return tokenizer, encoder, model, kb_config


In [ ]:
tokenizer, encoder, model, kb_config = load_kblam_model(
    model_path="/home/tbak/KBLaM/nano_model_v0/stage1_lr_0.0001KBTokenLayerFreq3MultiEntities2KeyFromkey_BigOAI_synthetic_data_phi3_step_2",
    encoder_path="/home/tbak/KBLaM/nano_model_v0/stage1_lr_0.0001KBTokenLayerFreq3MultiEntities2KeyFromkey_BigOAI_synthetic_data_phi3_step_2/encoder.pt",
    llm_type="phi3",
    encoder_spec="BigOAI",
    kb_layer_frequency=3,
    kb_scale_factor=None,
)


In [ ]:
from tqdm import tqdm
import torch
import numpy as np
from src.kblam.utils.train_utils import kb_to_embd

def load_transformed_kb_from_npy(encoder, key_npy_path, val_npy_path, device="cuda", batch_size=256):
    """
    Loads MiniLM .npy embeddings and transforms them using the encoder's adapters to match model shape.
    Returns:
        Tuple of (key_embs, val_embs) with shape (1, N, out_dim)
    """
    # Load and cast to float32
    key_embs = np.load(key_npy_path).astype(np.float32)
    val_embs = np.load(val_npy_path).astype(np.float32)

    assert key_embs.shape == val_embs.shape, "Key and value embeddings must match in shape"

    all_keys = []
    all_vals = []

    for i in tqdm(range(0, len(key_embs), batch_size), desc="Projecting embeddings"):
        key_batch = key_embs[i:i+batch_size]
        val_batch = val_embs[i:i+batch_size]

        # CPU → encoder handles conversion + projection internally
        keys_out, vals_out = kb_to_embd(
            encoder,
            precomputed_base_embd=np.stack([key_batch, val_batch])
        )

        all_keys.append(keys_out)
        all_vals.append(vals_out)

    keys_final = torch.cat(all_keys, dim=0).unsqueeze(0)  # Add batch dim
    vals_final = torch.cat(all_vals, dim=0).unsqueeze(0)

    return keys_final.to(device), vals_final.to(device)


In [ ]:
from transformers import AutoTokenizer
from src.kblam.models.phi3_model import KBLaMPhi3ForCausalLM
from src.kblam.kb_encoder import KBEncoder
from src.kblam.models.kblam_config import KBLaMConfig
from src.kblam.utils.eval_utils import answer_question
from pathlib import Path

# === Paths ===
model_path = "/home/tbak/KBLaM/nano_model_v0/stage1_lr_0.0001KBTokenLayerFreq3MultiEntities2KeyFromkey_BigOAI_synthetic_data_phi3_step_2"
encoder_path = f"{model_path}/encoder.pt"
keys_path = "/home/tbak/KBLaM/nano_data/synthetic_data_BigOAI_embd_key.npy"
vals_path = "/home/tbak/KBLaM/nano_data/synthetic_data_BigOAI_embd_value.npy"

# === Load tokenizer and model ===
tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-3-mini-4k-instruct", trust_remote_code=True)
tokenizer.pad_token = "^"

model = KBLaMPhi3ForCausalLM.from_pretrained(
    model_path,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
    local_files_only=True,
)
model.eval()

# === Load encoder and config ===
kb_config = KBLaMConfig(
    sep_query_head=True,
    kb_layer_frequency=3,
    kb_scale_factor=None,
)

encoder = KBEncoder(
    encoder_name="BigOAI",
    projector_type="linear",
    endpoint_url="",
    out_dim=model.config.hidden_size * (model.config.num_hidden_layers // kb_config.kb_layer_frequency + 1),
    frozen_base_model=True,
    projector_kwargs={"mlp_depth": 1, "mlp_hidden_dim": 512},
    device=torch.device("cuda"),
    get_oai_embd_online=False,  # we’re using cached .npy
)
encoder.load_state_dict(torch.load(encoder_path))

In [ ]:
# === Load and transform embeddings ===
kb_kvs = load_transformed_kb_from_npy(encoder, keys_path, vals_path)

# === Ask some test prompts ===
prompts = [
    "What are the key frequency indicators of bearing faults?",
    "How does gear mesh frequency help in diagnosing faults?",
    "What vibration patterns indicate mechanical looseness?",
]

for prompt in prompts:
    print(f"\nQ: {prompt}")
    response = answer_question(
        tokenizer,
        model,
        prompt,
        kb=kb_kvs,
        kb_config=kb_config,
    )
    print("A:", response)

In [ ]:
print("Expected out_dim:", model.config.hidden_size * (model.config.num_hidden_layers // kb_config.kb_layer_frequency + 1))
print("Loaded key_embs shape:", key_embs.shape)
print("Loaded value_embs shape:", val_embs.shape)
